In [3]:
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from typing import List, Tuple
import re
from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer
from src.gpt import KilterGPT

run_name = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = f"models/climb_gpt/{run_name}"
device = "cuda" if torch.cuda.is_available() else "cpu"

dp = DataPreprocessing()
datasets = dp.load_climbs()

# 80, 10, 10 split
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
    }

tokenizer = train_tokenizer(datasets, OUT_DIR)
datasets = dp.preprocess_datasets(datasets, tokenizer)

model = KilterGPT(
            vocab_size=tokenizer.vocab_size,
            n_embd=256,    
            n_head=4,      
            n_layer=6,     
            n_positions=128,
            dropout=0.1
        )

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="best",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=1000,
    num_train_epochs=1,  ####
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    # report_to="tensorboard",
    report_to="none",
    remove_unused_columns=False,
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model.model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()

model.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"\n✓ Model saved to {OUT_DIR}")

test_results = trainer.evaluate(datasets["test"])
print(f"\n✓ Test Loss: {test_results['eval_loss']:.4f}")
print(f"✓ Test Perplexity: {np.exp(test_results['eval_loss']):.2f}")


Loaded 76992 routes from cache data/climbs_cleaned.csv
Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('finish1209', 575), ('hand1330', 1058), ('start1556', 1757), ('feet1118', 208), ('feet1320', 1016), ('finish1572', 1823), ('feet1450', 1332), ('feet1327', 1044), ('start1184', 473), ('hand1191', 502)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to models/climb_gpt/run_20251031_134314


Map: 100%|██████████| 7700/7700 [00:01<00:00, 6439.98 examples/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
1000,5.861500,5.326397
2000,5.096100,4.860364
3000,4.784500,4.663754



✓ Model saved to models/climb_gpt/run_20251031_134314
